---
title: "Chapter 33: Predicting household spend"
---

In November 2019, a single-person household in Alabama, reporting an income between $25,000 and $49,999, bought a 65-inch Samsung 4K television, a PlayStation 4 Pro, a Microsoft Surface tablet, an Apple Watch, a GoPro camera bundle, a trampoline, and a kayak. Fifty-eight items, spread across four days. The total came to $11,709.16.

For the three months before that, the same household had spent an average of $49.47 a month. Across the roughly four years this account appears in the data, its lifetime total is $16,237.77. One month accounts for 72 percent of everything it has ever bought on the platform.

Nothing in this household's survey answers, age bracket, income, education, state, explains it. There's no flag for it and no obvious trigger. It's one row among 1.85 million, sitting in the data whether or not anyone thinks to look for it.

Chapters 28 through 32 gave you the machinery for exactly this kind of question: how to turn a vague business problem into a five-component specification, how the nine-stage workflow moves data through a model and into production, how scikit-learn's Transformer and Predictor interfaces work, how a Pipeline strings them together without leaking information across a split. You used all of it on hotel booking data that was already clean enough to teach with.

Here, you use it on a dataset nobody cleaned for teaching: five years of real Amazon purchase histories from 5,027 U.S. households, linked to a demographic and lifestyle survey each household completed once. The household above is what that data looks like up close. It's also the first of seven case studies built the same way, one real dataset and one real business question each, so the habit you build in this chapter (reason through the ask, prototype in the notebook, promote what works into a tested module) is the one the other six will follow.

## 1. Problem framing

Here's the request, exactly as it would actually arrive. You're the data scientist on a retention team at a company sitting on data like this, purchase histories linked to a demographic survey, for a panel of customers. Your manager stops by your desk and says:

"We want to know which customers are worth investing more in."

That's the whole brief. No target variable, no metric, no mention of a model. This is what a real ask sounds like, and your job right now is to figure out what it actually means, not to open a notebook.

### What does "worth investing in" even mean?

Before you touch a single file, you need to pull that sentence apart. It's collapsing at least four different problems into one, and they are not interchangeable:

- Customers who already spend the most. A ranking of the past, nothing predictive about it.
- Customers likely to spend more going forward. A forecast.
- Customers likely to leave if nobody intervenes. A churn problem.
- Customers who look like existing high spenders but haven't been identified yet. A segmentation problem.

Each one is a legitimate data science problem. Each one produces a different target variable, a different model, and a different action the business takes on your output. Guess wrong here and you could spend six months building the right system for the wrong question.

You go back and ask a follow-up question. It turns out "worth investing in" means deciding who gets priority for a loyalty program with a fixed, limited budget. The team wants customers ranked by how much they're likely to spend going forward, using the demographic and behavioural data already on file, before that spending happens. Not a scoreboard of who has already spent the most.

That answer resolves the first fork: this is a forecast of future value, not a description of past value, which rules out simply sorting by historical total. It also starts pinning down [Chapter 30's five-component spec](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem), one piece at a time, following wherever your own follow-up questions lead next:

<div class='ark-activity'>
<span class='ark-activity-title'><i class="bi bi-puzzle-fill"></i> Activity: write the spec card before reading further</span><br><br>
Using [Chapter 30's five-component spec](../04-ml-intro/30-ml-workflow.qmd#translating-a-business-problem-into-a-machine-learning-problem) (target variable, task type and paradigm, success metric, acceptable threshold, data specification), write your own answer for this problem before reading the discussion below. Compare where you landed once you get there. If your target variable differs, that's fine, it's the kind of decision Chapter 30's Pro Tip already warned about: agree it with the stakeholder who will actually use the model, because a model optimised toward the wrong objective will hit that objective and still miss the point.
</div>

**Target variable.** Not last month's spend, what you're predicting is next year's. You still need a window, a month, a year, a lifetime span all answer the question differently, and the right one depends on how often the loyalty program actually reallocates its budget. If the business revisits the list annually, that settles it: a household's spend over the coming twelve months.

**Task type and paradigm.** You're predicting a number from labelled historical examples where the outcome is already known. That's regression, supervised.

**Success metric.** The team isn't going to read predictions one row at a time, they're going to act on a ranked list. So the metric that matters is whether your model orders households correctly relative to each other, and whether the top decile it flags genuinely outspends everyone else. Mean absolute error is still worth tracking, mostly as a sanity check that the numbers themselves aren't nonsense.

**Acceptable threshold.** A model that can't beat "assume everyone spends the average" isn't worth shipping. Here's the real bar: households you rank in the top decile need to outspend a randomly chosen household by enough that the loyalty program's cost per member actually pays for itself, a number the business has to hand you, you can't derive it from the data alone.

**Data specification.** One demographic and lifestyle survey per household, answered once, and multi-year purchase history at the individual-transaction level, for 5,027 households.

::: {.callout-note collapse="true" icon=false}
## The resolved spec card

| Component | Specification |
|---|---|
| Target variable | Household's total spend over the coming 12 months |
| Task type and paradigm | Regression, supervised |
| Success metric | Rank quality (does the top decile of predicted spend actually outspend the rest), MAE as a secondary check |
| Acceptable threshold | Top-decile households must outspend a randomly chosen household by a margin the loyalty program's budget can justify |
| Data specification | One-time demographic and lifestyle survey plus multi-year, transaction-level purchase history, 5,027 U.S. households |
:::

That's the spec, and it took five follow-up questions to get there, not a target variable declared in the first paragraph. Every one of those questions is something a real stakeholder conversation would actually raise, and you'll ask versions of them again in every case study that follows.

Next: turning this into something durable enough that the rest of the project can't quietly drift from it, a Pydantic object, not a paragraph you could reinterpret three sections from now.